In [1]:
# ─────────────────────────────────────────────
# GENERADOR JSON FORMACIÓN PERMANENTE UPV
# ─────────────────────────────────────────────

!pip install -q beautifulsoup4 requests

import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


BASE_URL = "https://www.cfp.upv.es"


FUENTES = [
    (
        "cursos_online",
        "https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html"
    ),
    (
        "masters",
        "https://www.cfp.upv.es/formacion-permanente/masters/masters.html"
    )
]


# ─────────────────────────────────────────────
# LOCALIZAR CARPETA DEL PROGRAMA
# ─────────────────────────────────────────────

from google.colab import drive

drive.mount('/content/drive')


nombre_programa = "SacarJSON_FormacionPermanente.ipynb"  # pon aquí el nombre real

ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if nombre_programa in files:
        ruta_programa = root
        break


if ruta_programa is None:
    raise Exception(
        "No se ha encontrado la carpeta del programa"
    )


carpeta_json = os.path.join(
    ruta_programa,
    "JSONs"
)

os.makedirs(
    carpeta_json,
    exist_ok=True
)


SALIDA = os.path.join(
    carpeta_json,
    "formacion_permanente_upv.json"
)


HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (compatible; UPV-KB-Bot/1.0)"
}


PAUSA = 0.3



def get(url):

    try:
        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        r.raise_for_status()
        time.sleep(PAUSA)

        return BeautifulSoup(
            r.text,
            "html.parser"
        )

    except Exception as e:
        print("⚠️ error:", url, e)
        return None



def limpiar_nombre(t):

    t = re.sub(
        r"\s+",
        " ",
        t
    ).strip()

    return t



def clasificar_tipo(nombre):

    n = nombre.lower()

    if "máster" in n or "master" in n:
        return "Master"

    if "diploma de especialización" in n:
        return "Diploma de especialización"

    if "diploma de experto" in n:
        return "Diploma de experto"

    if "diploma de extensión" in n:
        return "Diploma de extensión"

    if "curso" in n:
        return "Curso"

    return "Otro"



def extraer_fichas(origen, url):

    soup = get(url)

    if not soup:
        return []


    encontrados = {}

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/formacion-permanente/curso/" not in href:
            continue


        url_final = urljoin(
            BASE_URL,
            href
        )


        nombre = limpiar_nombre(
            a.get_text(" ", strip=True)
        )

##############################################################
        # eliminar enlaces basura
        basura = [
            "Matriculable",
            "Más información",
            "Ver más",
            "Acceder",
            "Inscribirse"
        ]

        if any(
            b.lower() == nombre.lower()
            for b in basura
        ):
            continue
      ############################################################

        if len(nombre) < 5:
            continue


        encontrados[url_final] = {
            "nombre": nombre,
            "url": url_final,
            "origen": origen
        }


    return list(encontrados.values())



def extraer_detalle(item):

    soup = get(item["url"])

    if not soup:
        return item


    texto = soup.get_text(
        "\n",
        strip=True
    )


    # tipo
    item["tipo"] = clasificar_tipo(
        item["nombre"]
    )


    # ECTS
    m = re.search(
        r"(\d+(?:,\d+)?)\s*ECTS",
        texto,
        re.I
    )

    item["ects"] = (
        m.group(1)
        if m
        else None
    )


    # horas
    m = re.search(
        r"(\d+)\s*horas",
        texto,
        re.I
    )

    item["horas"] = (
        m.group(1)
        if m
        else None
    )


    # precio
    m = re.search(
        r"(\d+[.,]?\d*)\s*€",
        texto
    )

    item["precio"] = (
        m.group(1) + " €"
        if m
        else None
    )


    # campus
    campus = [
        "Vera",
        "Alcoy",
        "Gandia"
    ]

    item["campus"] = None

    for c in campus:
        if c.lower() in texto.lower():
            item["campus"] = c
            break


    # responsable/promotor
    claves = [
        "Promueve",
        "Organiza",
        "Responsable",
        "Director"
    ]

    item["promotor"] = None
    item["responsable"] = None


    for linea in texto.split("\n"):

        for clave in claves:

            if linea.lower().startswith(
                clave.lower()
            ):

                valor = linea.split(
                    ":",
                    1
                )

                if len(valor)==2:

                    if clave in ["Promueve","Organiza"]:
                        item["promotor"] = valor[1].strip()

                    else:
                        item["responsable"] = valor[1].strip()



    return item




# ─────────────────────────────────────────────
# EJECUCIÓN
# ─────────────────────────────────────────────


catalogo = {}



for origen, url in FUENTES:

    print("\nProcesando:", origen)

    fichas = extraer_fichas(
        origen,
        url
    )

    print(
        "Encontradas:",
        len(fichas)
    )


    for f in fichas:

        # deduplicación por URL
        catalogo[f["url"]] = f



print(
    "\nTotal sin duplicados:",
    len(catalogo)
)



resultado = []


for i,item in enumerate(
    catalogo.values(),
    1
):

    print(
        f"[{i}/{len(catalogo)}]",
        item["nombre"]
    )

    item = extraer_detalle(item)


    # id para nombre de archivo posterior
    item["id"] = re.sub(
        r"[^a-z0-9]+",
        "_",
        item["nombre"].lower()
    ).strip("_")


    resultado.append(item)



datos_finales = {
    "fuente": "Formación Permanente UPV",
    "total": len(resultado),
    "formaciones": resultado
}



os.makedirs(
    os.path.dirname(SALIDA),
    exist_ok=True
)


with open(
    SALIDA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        datos_finales,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\n✅ JSON generado:")
print(SALIDA)
print(
    "Elementos:",
    len(resultado)
)

Mounted at /content/drive

Procesando: cursos_online
Encontradas: 152

Procesando: masters
Encontradas: 136

Total sin duplicados: 288
[1/288] ANÁLISIS DE LA COYUNTURA ECONÓMICA
[2/288] MODELOS MULTICRITERIO APLICADOS A LA GESTIÓN DE CARTERAS
[3/288] GESTIÓN DE CARTERAS II
[4/288] INCENDIOS DE ORIGEN ELÉCTRICO EN EL ÁMBITO DOMÉSTICO. CAUSAS, RIESGOS Y ACTUACIÓN
[5/288] CAMPOS MAGNÉTICOS EN INSTALACIONES ELÉCTRICAS Y ALREDEDORES, Y SU CÁLCULO Y REPRESENTACIÓN CON CRMAG PLUS
[6/288] ANÁLISIS Y DISEÑO DE PUESTAS A TIERRA EN INSTALACIONES ELÉCTRICAS CON CRGROUND®
[7/288] ASESOR FINANCIERO
[8/288] AGENTE FINANCIERO EUROPEO
[9/288] ASISTENTE FINANCIERO EUROPEO
[10/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN ASESORÍA FINANCIERA 2025
[11/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN CRÉDITO INMOBILIARIO 2025
[12/288] ASESOR FINANCIERO EN CRÉDITO HIPOTECARIO
[13/288] INFORMADOR FINANCIERO EN CRÉDITO HIPOTECARIO
[14/288] CLOUD COMPUTING CON AMAZON WEB SERVICES (AWS)
[15/288] BASES DE DATOS ESPACIALES: POSTG